In [ ]:
# %pip install torch transformers
# %pip install peft
# %pip install trl


Defaulting to user installation because normal site-packages is not writeable
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.2/31.2 MB 17.8 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [trl]32m18/19 [trl]sets]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade 

In [1]:
from datasets import Dataset

data = [
    {
        "instruction": "What is Python?",
        "response": "Python is a high-level programming language."
    },
    {
        "instruction": "What is RAG?",
        "response": "RAG stands for Retrieval-Augmented Generation."
    },
    {
        "instruction": "What is a vector database?",
        "response": "A vector database stores and searches vector embeddings."
    }
]

dataset = Dataset.from_list(data)

/Users/neelamahlawat/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/neelamahlawat/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer

# Base model
# step1: Load the base model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Tokenizer
# step2: Load the tokenizer. IT loads the tokenizer that belongs to your selected LLM.
# It Load the tokenizer vocabulary and configuration that were created for this model.
# it  load the tokenizer from the model name specified in the model_name variable. 
# The tokenizer is responsible for converting text into a format that the model can understand, such as token IDs. 
# If the tokenizer does not have a pad token, it sets the pad token to be the same as the end-of-sequence (eos) token.
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Tokenizer loaded from LLM", tokenizer)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#Load model
# step3: Load the model. It loads the pre-trained model weights and configuration from the specified model name.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() #if the GPU supports bfloat16 precision, it uses that; otherwise, it falls back to float16 precision.
    else torch.float16,
    device_map="auto" # if set to "auto", the model will automatically be placed on the available GPU(s) or CPU based on the system's configuration and available resources.
)

# LoRA configuration
# step4: Configure LoRA. It sets up the configuration for LoRA (Low-Rank Adaptation) training.
# LoRA is a technique that allows for efficient fine-tuning of large language models by introducing low-rank adaptation matrices.
#low rank means that the adaptation matrices have a lower rank than the original model's weight matrices, which reduces the number of parameters that need to be learned during fine-tuning.
# we freeze the original model's parameters and only train the low-rank adaptation matrices, which significantly reduces the number of trainable parameters and speeds up training.
# It specifies the hyperparameters for LoRA, including the rank (r), alpha (lora_alpha), dropout rate (lora_dropout), bias handling, and the task type (CAUSAL_LM).
lora_config = LoraConfig(
    r=16, #it specifies the rank of the low-rank adaptation matrices. A higher rank allows for more expressive adaptations but increases the number of parameters.
    lora_alpha=32, #it is a scaling factor that controls the contribution of the LoRA adaptation to the model's output. A higher alpha value increases the influence of the LoRA adaptation.
    lora_dropout=0.05, #it specifies the dropout rate applied to the LoRA adaptation during training. Dropout is a regularization technique that helps prevent overfitting by randomly dropping out a fraction of the adaptation during training.
    bias="none", #it specifies how biases are handled in the LoRA adaptation. In this case, "none" means that biases are not adapted using LoRA.
    task_type="CAUSAL_LM", #it specifies the type of task for which the LoRA adaptation is being applied. In this case, "CAUSAL_LM" indicates that the model is being fine-tuned for a causal language modeling task, where the model predicts the next token in a sequence based on the previous tokens.

    target_modules=[ 
        "q_proj", #it specifies the names of the modules in the model that will be adapted using LoRA. In this case, it includes the query projection ("q_proj"), key projection ("k_proj"), value projection ("v_proj"), and output projection ("o_proj") layers of the model's attention mechanism. These are the layers where LoRA will be applied to learn low-rank adaptations.
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

#Training arguments
# step5: Set training arguments. It defines the training parameters for fine-tuning the model using LoRA.
training_args = TrainingArguments(
    output_dir="./lora_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none"
)

# Trainer
# step6: Create the SFTTrainer. It initializes the SFTTrainer, which is responsible for managing the training process.
# It takes the model, training dataset, LoRA configuration, and training arguments as inputs. 
# The SFTTrainer handles the training loop, optimization, and evaluation of the model during fine-tuning.
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args
)

# Train
# step7: Start training. It starts the training process by calling the train() method of the SFTTrainer.
trainer.train()

# Save LoRA adapter
# step8: Save the trained LoRA adapter. It saves the trained LoRA adapter to the specified output directory ("./lora_model").
trainer.save_model("./lora_model") # it saves the model weights and configuration of the LoRA adapter, allowing you to load and use the fine-tuned model later.
tokenizer.save_pretrained("./lora_model") # it saves the tokenizer configuration and vocabulary to the same output directory, ensuring that you can use the same tokenizer when loading the fine-tuned model for inference.

/Users/neelamahlawat/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/neelamahlawat/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
